# Lesson 28 Lab — Performance Regression CI

**Puzzle:** When golden outputs, sample distributions, version pins, and warning/error gates change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates golden outputs, sample distributions, version pins, and warning/error gates and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

A kernel CI job needs correctness and performance contracts. Correctness uses adversarial shapes and golden references; performance compares stable sample summaries on controlled hardware. Version identity travels with the baseline because compiler and backend changes can alter generated code.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["golden outputs, sample distributions, version pins, and warning/error gates"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

A single noisy sample or a benchmark retained only on a developer laptop cannot support an auditable regression decision.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 28
LESSON_TITLE = 'Performance Regression CI'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260841
}


## 5. Freeze the experiment

**Experiment:** Collect two independent 30-sample groups and classify their median ratio with frozen warning and error gates.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 0.9753381497048288,
  "secondary": "pass",
  "max_abs_error": 4.76837158203125e-07,
  "passed": true,
  "details": {
    "baseline_ms": 0.020111999474465847,
    "candidate_ms": 0.01961600035429001,
    "warning_ratio": 1.1,
    "error_ratio": 1.2,
    "baseline_samples_ms": [
      0.030880000442266464,
      0.024320000782608986,
      0.022016000002622604,
      0.021663999184966087,
      0.01958400011062622,
      0.020959999412298203,
      0.022175999358296394,
      0.02112000063061714,
      0.02147199958562851,
      0.02112000063061714,
      0.02022399939596653,
      0.019360000267624855,
      0.021376000717282295,
      0.019807999953627586,
      0.0208320003002882,
      0.01852799952030182,
      0.019999999552965164,
      0.018303999677300453,
      0.018751999363303185,
      0.02035200037062168,
      0.01945599913597107,
      0.02147199958562851,
      0.01961600035429001,
      0.018464000895619392,
      0.01961600035429001,
      0.018239999189

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Candidate/baseline ratio | 0.975x |
| CI status | pass |
| Maximum absolute error | 4.768e-07 |
| Acceptance gate | true |


## 8. Explain without overclaiming

Two independent warm sample groups produced a 0.975x candidate/baseline ratio, classified pass under frozen 1.10/1.20 warning/error gates.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Use CI as a triage signal: reproduce warnings, block clear regressions, and rebaseline only with an explained environment change.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 28,
  "title": "Performance Regression CI",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260841
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 0.9753381497048288,
    "secondary": "pass",
    "max_abs_error": 4.76837158203125e-07,
    "passed": true,
    "details": {
      "baseline_ms": 0.020111999474465847,
      "candidate_ms": 0.01961600035429001,
      "warning_ratio": 1.1,
      "error_ratio": 1.2,
      "baseline_samples_ms": [
        0.030880000442266464,
        0.024320000782608986,
        0.022016000002622604,
        0.021663999184966087,
        0.01958400011062622,
        0.020959999412298203,
        0.022175999358296394,
        0.02112000063061714,
        0.02147199958562851,
        0.021120

## 10. Make the bounded decision

> Use CI as a triage signal: reproduce warnings, block clear regressions, and rebaseline only with an explained environment change.

**Failure analysis:** A single noisy sample or a benchmark retained only on a developer laptop cannot support an auditable regression decision.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
